In [0]:
# 1. Define the catalog and schemas
catalog = "gpo_analytics"
silver_schema = "silver"
gold_schema = "gold"

# 2. Create the Gold schema if it does not exist
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{gold_schema}")

# 3. Load the Silver study table
study_silver = spark.table(
    f"{catalog}.{silver_schema}.study"
)

# 4. Display the study table structure
study_silver.printSchema()

# 5. Preview five study records
display(study_silver.limit(5))

root
 |-- StudyID: string (nullable = true)
 |-- ProtocolID: string (nullable = true)
 |-- StudyName: string (nullable = true)
 |-- Phase: string (nullable = true)
 |-- TherapeuticArea: string (nullable = true)
 |-- Status: string (nullable = true)
 |-- PlannedEnrollment: integer (nullable = true)
 |-- StudyStartDate: date (nullable = true)
 |-- StudyEndDate: date (nullable = true)



StudyID,ProtocolID,StudyName,Phase,TherapeuticArea,Status,PlannedEnrollment,StudyStartDate,StudyEndDate
ST001,GPO-ONC-101,Oncology Clinical Study 01,Phase I,Oncology,Recruiting,162,2025-04-28,2027-12-19
ST002,GPO-IMM-102,Immunology Clinical Study 02,Phase II,Immunology,Recruiting,331,2024-04-19,2028-07-30
ST003,GPO-RAR-103,Rare Disease Clinical Study 03,Phase I,Rare Disease,Active,161,2025-02-03,2026-08-09
ST004,GPO-ONC-104,Oncology Clinical Study 04,Phase I,Oncology,Recruiting,165,2024-07-31,2028-12-05
ST005,GPO-RAR-105,Rare Disease Clinical Study 05,Phase I,Rare Disease,Recruiting,473,2025-04-14,2028-02-26


In [0]:
from pyspark.sql import functions as F

# 1. Load the existing Silver tables
studies = spark.table("gpo_analytics.silver.study")
subjects = spark.table("gpo_analytics.silver.subject")
sites = spark.table("gpo_analytics.silver.site")

# 2. Calculate cumulative enrollment for each study
enrollment = (
    subjects
    .filter(
        F.col("SubjectStatus").isin(
            "Enrolled", "Active", "Completed", "Withdrawn"
        )
    )
    .groupBy("StudyID")
    .agg(
        F.countDistinct("SubjectID").alias("ActualEnrollment")
    )
)

# 3. Count distinct sites associated with each study
site_counts = (
    sites
    .groupBy("StudyID")
    .agg(
        F.countDistinct("SiteID").alias("SiteCount")
    )
)

# 4. Combine study details, enrollment, and site counts
study_kpis = (
    studies
    .join(enrollment, on="StudyID", how="left")
    .join(site_counts, on="StudyID", how="left")
    .fillna({
        "ActualEnrollment": 0,
        "SiteCount": 0
    })
)

# 5. Calculate enrollment progress as a percentage
study_kpis = study_kpis.withColumn(
    "EnrollmentProgressPct",
    F.when(
        F.col("PlannedEnrollment").isNotNull()
        & (F.col("PlannedEnrollment") > 0),
        F.round(
            F.col("ActualEnrollment") * 100
            / F.col("PlannedEnrollment"),
            2
        )
    ).otherwise(None)
)

# 6. Select the final Gold table columns
study_kpis = study_kpis.select(
    "StudyID",
    "ProtocolID",
    "StudyName",
    "Phase",
    "TherapeuticArea",
    "Status",
    "PlannedEnrollment",
    "ActualEnrollment",
    "EnrollmentProgressPct",
    "SiteCount",
    "StudyStartDate",
    "StudyEndDate"
)

# 7. Create the Gold Delta table
study_kpis.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gpo_analytics.gold.study_kpis")

# 8. Display the created table
display(
    spark.table("gpo_analytics.gold.study_kpis")
    .orderBy("StudyID")
)

StudyID,ProtocolID,StudyName,Phase,TherapeuticArea,Status,PlannedEnrollment,ActualEnrollment,EnrollmentProgressPct,SiteCount,StudyStartDate,StudyEndDate
ST001,GPO-ONC-101,Oncology Clinical Study 01,Phase I,Oncology,Recruiting,162,74,45.68,3,2025-04-28,2027-12-19
ST002,GPO-IMM-102,Immunology Clinical Study 02,Phase II,Immunology,Recruiting,331,145,43.81,6,2024-04-19,2028-07-30
ST003,GPO-RAR-103,Rare Disease Clinical Study 03,Phase I,Rare Disease,Active,161,106,65.84,4,2025-02-03,2026-08-09
ST004,GPO-ONC-104,Oncology Clinical Study 04,Phase I,Oncology,Recruiting,165,95,57.58,4,2024-07-31,2028-12-05
ST005,GPO-RAR-105,Rare Disease Clinical Study 05,Phase I,Rare Disease,Recruiting,473,82,17.34,3,2025-04-14,2028-02-26
ST006,GPO-RAR-106,Rare Disease Clinical Study 06,Phase II,Rare Disease,Active,498,88,17.67,4,2024-07-16,2026-05-21
ST007,GPO-CAR-107,Cardiology Clinical Study 07,Phase III,Cardiology,Recruiting,523,149,28.49,7,2024-05-08,2027-07-03
ST008,GPO-IMM-108,Immunology Clinical Study 08,Phase III,Immunology,Active,298,45,15.1,2,2023-07-20,2028-10-12
ST009,GPO-IMM-109,Immunology Clinical Study 09,Phase II,Immunology,Completed,495,169,34.14,7,2024-12-06,2027-03-18
ST010,GPO-ONC-110,Oncology Clinical Study 10,Phase I,Oncology,Recruiting,515,174,33.79,8,2024-08-20,2027-05-01


In [0]:
from pyspark.sql import functions as F

# 1. Load the Gold table and its Silver source tables
gold = spark.table("gpo_analytics.gold.study_kpis")
subjects = spark.table("gpo_analytics.silver.subject")
sites = spark.table("gpo_analytics.silver.site")

# 2. Check total rows and unique study IDs
print("Gold row count:", gold.count())
print(
    "Distinct StudyID count:",
    gold.select("StudyID").distinct().count()
)
print(
    "Missing StudyID:",
    gold.filter(
        F.col("StudyID").isNull()
        | (F.trim(F.col("StudyID")) == "")
    ).count()
)

# 3. Check whether any study IDs are duplicated
print(
    "Duplicate StudyID extra rows:",
    gold.count() - gold.select("StudyID").distinct().count()
)

# 4. Recalculate enrollment using the same provisional rule
expected_enrollment = (
    subjects
    .filter(
        F.col("SubjectStatus").isin(
            "Enrolled", "Active", "Completed", "Withdrawn"
        )
    )
    .groupBy("StudyID")
    .agg(
        F.countDistinct("SubjectID").alias("ExpectedEnrollment")
    )
)

enrollment_check = (
    gold
    .join(expected_enrollment, on="StudyID", how="left")
    .filter(
        F.col("ActualEnrollment")
        != F.coalesce(F.col("ExpectedEnrollment"), F.lit(0))
    )
)

print(
    "Studies with incorrect ActualEnrollment:",
    enrollment_check.count()
)

# 5. Recalculate site counts
expected_sites = (
    sites
    .groupBy("StudyID")
    .agg(F.countDistinct("SiteID").alias("ExpectedSiteCount"))
)

site_check = (
    gold
    .join(expected_sites, on="StudyID", how="left")
    .filter(
        F.col("SiteCount")
        != F.coalesce(F.col("ExpectedSiteCount"), F.lit(0))
    )
)

print("Studies with incorrect SiteCount:", site_check.count())

# 6. Check the enrollment percentage calculation
progress_check = gold.filter(
    (F.col("PlannedEnrollment") > 0)
    & (
        F.abs(
            F.col("EnrollmentProgressPct")
            - F.round(
                F.col("ActualEnrollment") * 100
                / F.col("PlannedEnrollment"),
                2
            )
        ) > 0.001
    )
)

print(
    "Studies with incorrect EnrollmentProgressPct:",
    progress_check.count()
)

# 7. Review the final schema
gold.printSchema()

Gold row count: 20
Distinct StudyID count: 20
Missing StudyID: 0
Duplicate StudyID extra rows: 0
Studies with incorrect ActualEnrollment: 0
Studies with incorrect SiteCount: 0
Studies with incorrect EnrollmentProgressPct: 0
root
 |-- StudyID: string (nullable = true)
 |-- ProtocolID: string (nullable = true)
 |-- StudyName: string (nullable = true)
 |-- Phase: string (nullable = true)
 |-- TherapeuticArea: string (nullable = true)
 |-- Status: string (nullable = true)
 |-- PlannedEnrollment: integer (nullable = true)
 |-- ActualEnrollment: long (nullable = true)
 |-- EnrollmentProgressPct: double (nullable = true)
 |-- SiteCount: long (nullable = true)
 |-- StudyStartDate: date (nullable = true)
 |-- StudyEndDate: date (nullable = true)



In [0]:
from pyspark.sql import functions as F

adverse_events = spark.table(
    "gpo_analytics.silver.adverseevents"
)

print("Adverse event row count:", adverse_events.count())

display(
    adverse_events.groupBy("Severity")
    .count()
    .orderBy("Severity")
)

display(
    adverse_events.groupBy("Serious")
    .count()
    .orderBy("Serious")
)

Adverse event row count: 1200


Severity,count
Grade 1,537
Grade 2,360
Grade 3,205
Grade 4,69
Grade 5,29


Serious,count
No,636
Yes,564


In [0]:
from pyspark.sql import functions as F

# 1. Load the Silver source tables
adverse_events = spark.table(
    "gpo_analytics.silver.adverseevents"
)

subjects = spark.table(
    "gpo_analytics.silver.subject"
)

# 2. Connect each adverse event to its study
events_with_study = (
    adverse_events
    .join(
        subjects.select("SubjectID", "StudyID"),
        on="SubjectID",
        how="left"
    )
)

# 3. Group the events by study and calculate safety metrics
safety_summary = (
    events_with_study
    .groupBy("StudyID")
    .agg(
        F.count("AEID").alias("TotalAdverseEvents"),

        F.sum(
            F.when(F.col("Serious") == "Yes", 1).otherwise(0)
        ).alias("SeriousEvents"),

        F.sum(
            F.when(F.col("Severity") == "Grade 1", 1).otherwise(0)
        ).alias("Grade1Events"),

        F.sum(
            F.when(F.col("Severity") == "Grade 2", 1).otherwise(0)
        ).alias("Grade2Events"),

        F.sum(
            F.when(F.col("Severity") == "Grade 3", 1).otherwise(0)
        ).alias("Grade3Events"),

        F.sum(
            F.when(F.col("Severity") == "Grade 4", 1).otherwise(0)
        ).alias("Grade4Events"),

        F.sum(
            F.when(F.col("Severity") == "Grade 5", 1).otherwise(0)
        ).alias("Grade5Events")
    )
)

# 4. Save the result as a Gold Delta table
safety_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gpo_analytics.gold.safety_summary")

# 5. Display the saved table
display(
    spark.table("gpo_analytics.gold.safety_summary")
    .orderBy("StudyID")
)

StudyID,TotalAdverseEvents,SeriousEvents,Grade1Events,Grade2Events,Grade3Events,Grade4Events,Grade5Events
ST001,42,20,21,12,6,3,0
ST002,71,33,36,20,11,4,0
ST003,51,20,22,17,8,3,1
ST004,49,27,23,13,11,1,1
ST005,38,18,13,11,8,4,2
ST006,55,21,21,18,13,3,0
ST007,94,45,42,30,11,5,6
ST008,20,9,6,6,4,3,1
ST009,81,38,34,24,15,5,3
ST010,87,42,48,22,10,4,3


In [0]:
from pyspark.sql import functions as F

# 1. Load the Gold table and Silver source tables
gold = spark.table("gpo_analytics.gold.safety_summary")
events = spark.table("gpo_analytics.silver.adverseevents")
subjects = spark.table("gpo_analytics.silver.subject")

# 2. Check the number of studies and duplicate IDs
print("Gold row count:", gold.count())
print(
    "Distinct StudyID count:",
    gold.select("StudyID").distinct().count()
)
print(
    "Duplicate StudyID extra rows:",
    gold.count() - gold.select("StudyID").distinct().count()
)

# 3. Check total events before and after the study join
events_with_study = events.join(
    subjects.select("SubjectID", "StudyID"),
    on="SubjectID",
    how="left"
)

print("Silver adverse-event count:", events.count())
print("Events after subject join:", events_with_study.count())
print("Gold total adverse events:", gold.agg(
    F.sum("TotalAdverseEvents")
).first()[0])

# 4. Recalculate the safety metrics independently
expected = (
    events_with_study
    .groupBy("StudyID")
    .agg(
        F.count("AEID").alias("ExpectedTotal"),
        F.sum(
            F.when(F.col("Serious") == "Yes", 1).otherwise(0)
        ).alias("ExpectedSerious"),
        *[
            F.sum(
                F.when(
                    F.col("Severity") == f"Grade {grade}", 1
                ).otherwise(0)
            ).alias(f"ExpectedGrade{grade}")
            for grade in range(1, 6)
        ]
    )
)

comparison = gold.join(expected, on="StudyID", how="full")

# 5. Count studies where any metric differs
checks = [
    ("TotalAdverseEvents", "ExpectedTotal"),
    ("SeriousEvents", "ExpectedSerious"),
    ("Grade1Events", "ExpectedGrade1"),
    ("Grade2Events", "ExpectedGrade2"),
    ("Grade3Events", "ExpectedGrade3"),
    ("Grade4Events", "ExpectedGrade4"),
    ("Grade5Events", "ExpectedGrade5")
]

mismatch_condition = F.lit(False)

for actual_col, expected_col in checks:
    mismatch_condition = mismatch_condition | (
        ~F.col(actual_col).eqNullSafe(F.col(expected_col))
    )

print(
    "Studies with mismatched safety metrics:",
    comparison.filter(mismatch_condition).count()
)

# 6. Review the Gold schema
gold.printSchema()

Gold row count: 20
Distinct StudyID count: 20
Duplicate StudyID extra rows: 0
Silver adverse-event count: 1200
Events after subject join: 1200
Gold total adverse events: 1200
Studies with mismatched safety metrics: 0
root
 |-- StudyID: string (nullable = true)
 |-- TotalAdverseEvents: long (nullable = true)
 |-- SeriousEvents: long (nullable = true)
 |-- Grade1Events: long (nullable = true)
 |-- Grade2Events: long (nullable = true)
 |-- Grade3Events: long (nullable = true)
 |-- Grade4Events: long (nullable = true)
 |-- Grade5Events: long (nullable = true)



In [0]:
from pyspark.sql import functions as F

subjects = spark.table("gpo_analytics.silver.subject")

print("Total subjects:", subjects.count())

print(
    "Subjects with an enrollment date:",
    subjects.filter(F.col("EnrollmentDate").isNotNull()).count()
)

display(
    subjects.groupBy("SubjectStatus")
    .agg(
        F.count("*").alias("SubjectCount"),
        F.count("EnrollmentDate").alias("SubjectsWithEnrollmentDate")
    )
    .orderBy("SubjectStatus")
)

display(
    subjects
    .filter(F.col("EnrollmentDate").isNotNull())
    .groupBy(F.date_format("EnrollmentDate", "yyyy-MM").alias("EnrollmentMonth"))
    .count()
    .orderBy("EnrollmentMonth")
)

Total subjects: 3000
Subjects with an enrollment date: 2394


SubjectStatus,SubjectCount,SubjectsWithEnrollmentDate
Active,1160,1160
Completed,484,484
Enrolled,520,520
Screen Failed,315,0
Screened,291,0
Withdrawn,230,230


EnrollmentMonth,count
2023-06,30
2023-07,64
2023-08,58
2023-09,64
2023-10,62
2023-11,65
2023-12,60
2024-01,65
2024-02,32
2024-03,58


In [0]:
from pyspark.sql import functions as F

# 1. Load the existing Silver subject table
subjects = spark.table("gpo_analytics.silver.subject")

# 2. Keep subjects included in the cumulative enrollment metric
eligible_subjects = subjects.filter(
    F.col("SubjectStatus").isin(
        "Enrolled", "Active", "Completed", "Withdrawn"
    )
)

# 3. Keep records that have an enrollment date
dated_subjects = eligible_subjects.filter(
    F.col("EnrollmentDate").isNotNull()
)

# 4. Group subjects by study and enrollment month
enrollment_trends = (
    dated_subjects
    .groupBy(
        "StudyID",
        F.date_trunc("month", F.col("EnrollmentDate")).cast("date")
        .alias("EnrollmentMonth")
    )
    .agg(
        F.countDistinct("SubjectID").alias("NewEnrollments")
    )
)

# 5. Save the result as a Gold Delta table
enrollment_trends.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gpo_analytics.gold.enrollment_trends")

# 6. Display the saved results
display(
    spark.table("gpo_analytics.gold.enrollment_trends")
    .orderBy("StudyID", "EnrollmentMonth")
)

StudyID,EnrollmentMonth,NewEnrollments
ST001,2023-07-01,4
ST001,2023-08-01,2
ST001,2023-10-01,3
ST001,2023-11-01,1
ST001,2023-12-01,6
ST001,2024-01-01,1
ST001,2024-02-01,1
ST001,2024-03-01,4
ST001,2024-04-01,3
ST001,2024-05-01,4


In [0]:
from pyspark.sql import functions as F

# 1. Load the saved Gold table and its Silver source
gold = spark.table("gpo_analytics.gold.enrollment_trends")
subjects = spark.table("gpo_analytics.silver.subject")

# 2. Rebuild the expected monthly counts from Silver
expected = (
    subjects
    .filter(
        F.col("SubjectStatus").isin(
            "Enrolled", "Active", "Completed", "Withdrawn"
        )
        & F.col("EnrollmentDate").isNotNull()
    )
    .groupBy(
        "StudyID",
        F.date_trunc(
            "month", F.col("EnrollmentDate")
        ).cast("date").alias("EnrollmentMonth")
    )
    .agg(
        F.countDistinct("SubjectID").alias("ExpectedNewEnrollments")
    )
)

# 3. Check row counts and total monthly enrollments
print("Gold row count:", gold.count())
print("Expected row count:", expected.count())
print("Gold total new enrollments:", gold.agg(
    F.sum("NewEnrollments")
).first()[0])
print("Expected total new enrollments:", expected.agg(
    F.sum("ExpectedNewEnrollments")
).first()[0])

# 4. Check duplicate study-month combinations
print(
    "Duplicate study-month rows:",
    gold.count()
    - gold.select("StudyID", "EnrollmentMonth").distinct().count()
)

# 5. Compare saved values against independently recalculated values
comparison = (
    gold.alias("g")
    .join(
        expected.alias("e"),
        on=["StudyID", "EnrollmentMonth"],
        how="full"
    )
)

mismatches = comparison.filter(
    ~F.col("NewEnrollments").eqNullSafe(
        F.col("ExpectedNewEnrollments")
    )
)

print("Mismatched study-month rows:", mismatches.count())

# 6. Show the saved table's schema
gold.printSchema()

Gold row count: 745
Expected row count: 745
Gold total new enrollments: 2394
Expected total new enrollments: 2394
Duplicate study-month rows: 0
Mismatched study-month rows: 0
root
 |-- StudyID: string (nullable = true)
 |-- EnrollmentMonth: date (nullable = true)
 |-- NewEnrollments: long (nullable = true)



In [0]:
from pyspark.sql import functions as F

sites = spark.table("gpo_analytics.silver.site")
subjects = spark.table("gpo_analytics.silver.subject")

print("Silver site rows:", sites.count())
print("Silver subject rows:", subjects.count())

display(
    sites.groupBy("SiteStatus")
    .count()
    .orderBy("SiteStatus")
)

display(
    subjects.groupBy("SubjectStatus")
    .count()
    .orderBy("SubjectStatus")
)

display(
    sites.select("SiteID", "StudyID", "SiteName", "Country", "SiteStatus")
    .orderBy("StudyID", "SiteID")
    .limit(10)
)

Silver site rows: 100
Silver subject rows: 3000


SiteStatus,count
Active,68
Closed,15
Initiated,12
On Hold,5


SubjectStatus,count
Active,1160
Completed,484
Enrolled,520
Screen Failed,315
Screened,291
Withdrawn,230


SiteID,StudyID,SiteName,Country,SiteStatus
SITE030,ST001,Madrid Clinical Research Site 030,Spain,Active
SITE045,ST001,Boston Clinical Research Site 045,USA,Active
SITE088,ST001,Toronto Clinical Research Site 088,Canada,Active
SITE006,ST002,New York Clinical Research Site 006,USA,Initiated
SITE015,ST002,Tokyo Clinical Research Site 015,Japan,Active
SITE035,ST002,Hyderabad Clinical Research Site 035,India,Closed
SITE038,ST002,London Clinical Research Site 038,UK,Active
SITE080,ST002,Sydney Clinical Research Site 080,Australia,Active
SITE093,ST002,Sydney Clinical Research Site 093,Australia,Active
SITE050,ST003,Sydney Clinical Research Site 050,Australia,Closed


In [0]:
from pyspark.sql import functions as F

# 1. Load the Silver source tables
sites = spark.table("gpo_analytics.silver.site")
subjects = spark.table("gpo_analytics.silver.subject")

# 2. Calculate subject counts for each site
subject_counts = (
    subjects
    .groupBy("SiteID")
    .agg(
        F.countDistinct("SubjectID").alias("TotalSubjects"),

        F.countDistinct(
            F.when(
                F.col("EnrollmentDate").isNotNull(),
                F.col("SubjectID")
            )
        ).alias("EnrolledSubjects"),

        F.countDistinct(
            F.when(
                F.col("SubjectStatus") == "Active",
                F.col("SubjectID")
            )
        ).alias("ActiveSubjects"),

        F.countDistinct(
            F.when(
                F.col("SubjectStatus") == "Completed",
                F.col("SubjectID")
            )
        ).alias("CompletedSubjects"),

        F.countDistinct(
            F.when(
                F.col("SubjectStatus") == "Screen Failed",
                F.col("SubjectID")
            )
        ).alias("ScreenFailedSubjects"),

        F.countDistinct(
            F.when(
                F.col("SubjectStatus") == "Withdrawn",
                F.col("SubjectID")
            )
        ).alias("WithdrawnSubjects")
    )
)

# 3. Keep every site and attach its subject counts
site_performance = (
    sites
    .join(subject_counts, on="SiteID", how="left")
    .fillna({
        "TotalSubjects": 0,
        "EnrolledSubjects": 0,
        "ActiveSubjects": 0,
        "CompletedSubjects": 0,
        "ScreenFailedSubjects": 0,
        "WithdrawnSubjects": 0
    })
    .select(
        "SiteID",
        "StudyID",
        "SiteName",
        "Country",
        "SiteStatus",
        "TotalSubjects",
        "EnrolledSubjects",
        "ActiveSubjects",
        "CompletedSubjects",
        "ScreenFailedSubjects",
        "WithdrawnSubjects"
    )
)

# 4. Save the Gold table
site_performance.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gpo_analytics.gold.site_performance")

# 5. Display the saved results
display(
    spark.table("gpo_analytics.gold.site_performance")
    .orderBy("StudyID", "SiteID")
)

SiteID,StudyID,SiteName,Country,SiteStatus,TotalSubjects,EnrolledSubjects,ActiveSubjects,CompletedSubjects,ScreenFailedSubjects,WithdrawnSubjects
SITE030,ST001,Madrid Clinical Research Site 030,Spain,Active,22,20,11,4,1,3
SITE045,ST001,Boston Clinical Research Site 045,USA,Active,40,32,16,5,5,1
SITE088,ST001,Toronto Clinical Research Site 088,Canada,Active,38,22,11,5,7,2
SITE006,ST002,New York Clinical Research Site 006,USA,Initiated,24,19,12,3,3,1
SITE015,ST002,Tokyo Clinical Research Site 015,Japan,Active,31,24,16,5,4,1
SITE035,ST002,Hyderabad Clinical Research Site 035,India,Closed,37,34,19,8,2,3
SITE038,ST002,London Clinical Research Site 038,UK,Active,31,20,12,2,6,1
SITE080,ST002,Sydney Clinical Research Site 080,Australia,Active,30,25,13,4,3,1
SITE093,ST002,Sydney Clinical Research Site 093,Australia,Active,32,23,14,3,6,1
SITE050,ST003,Sydney Clinical Research Site 050,Australia,Closed,34,28,14,4,3,3


In [0]:
from pyspark.sql import functions as F

# Load the Gold table we just created
site_gold = spark.table("gpo_analytics.gold.site_performance")

# Load the cleaned Silver tables used to create it
sites = spark.table("gpo_analytics.silver.site")
subjects = spark.table("gpo_analytics.silver.subject")

# Count the rows and unique site IDs
print("Gold site rows:", site_gold.count())
print("Silver site rows:", sites.count())
print("Distinct Gold SiteIDs:", site_gold.select("SiteID").distinct().count())

# Check for missing SiteIDs
print(
    "Missing SiteIDs:",
    site_gold.filter(
        F.col("SiteID").isNull() | (F.trim(F.col("SiteID")) == "")
    ).count()
)

# Check whether any SiteID appears more than once
print(
    "Duplicate SiteID extra rows:",
    site_gold.count() - site_gold.select("SiteID").distinct().count()
)

# Recalculate subject counts from Silver data
expected_counts = (
    subjects.groupBy("SiteID")
    .agg(
        F.count("*").alias("ExpectedTotalSubjects"),
       F.countDistinct(
    F.when(
        F.col("EnrollmentDate").isNotNull(),
        F.col("SubjectID")
    )
).alias("ExpectedEnrolledSubjects"),
        F.sum(F.when(F.col("SubjectStatus") == "Active", 1).otherwise(0))
            .alias("ExpectedActiveSubjects"),
        F.sum(F.when(F.col("SubjectStatus") == "Completed", 1).otherwise(0))
            .alias("ExpectedCompletedSubjects"),
        F.sum(F.when(F.col("SubjectStatus") == "Screen Failed", 1).otherwise(0))
            .alias("ExpectedScreenFailedSubjects"),
        F.sum(F.when(F.col("SubjectStatus") == "Withdrawn", 1).otherwise(0))
            .alias("ExpectedWithdrawnSubjects")
    )
)

# Compare the stored Gold counts with the recalculated counts
comparison = (
    site_gold.alias("g")
    .join(expected_counts.alias("e"), on="SiteID", how="left")
    .filter(
        (F.col("g.TotalSubjects") != F.coalesce(F.col("e.ExpectedTotalSubjects"), F.lit(0)))
        | (F.col("g.EnrolledSubjects") != F.coalesce(F.col("e.ExpectedEnrolledSubjects"), F.lit(0)))
        | (F.col("g.ActiveSubjects") != F.coalesce(F.col("e.ExpectedActiveSubjects"), F.lit(0)))
        | (F.col("g.CompletedSubjects") != F.coalesce(F.col("e.ExpectedCompletedSubjects"), F.lit(0)))
        | (F.col("g.ScreenFailedSubjects") != F.coalesce(F.col("e.ExpectedScreenFailedSubjects"), F.lit(0)))
        | (F.col("g.WithdrawnSubjects") != F.coalesce(F.col("e.ExpectedWithdrawnSubjects"), F.lit(0)))
    )
)

print("Sites with mismatched subject counts:", comparison.count())

Gold site rows: 100
Silver site rows: 100
Distinct Gold SiteIDs: 100
Missing SiteIDs: 0
Duplicate SiteID extra rows: 0
Sites with mismatched subject counts: 0


In [0]:
from pyspark.sql import functions as F

# Read the Gold site summary
site_gold = spark.table("gpo_analytics.gold.site_performance")

# Read the cleaned subject records
subjects = spark.table("gpo_analytics.silver.subject")

# Show the saved Gold counts for one site
display(
    site_gold
    .filter(F.col("SiteID") == "SITE030")
    .select(
        "SiteID",
        "TotalSubjects",
        "EnrolledSubjects",
        "ActiveSubjects",
        "CompletedSubjects",
        "ScreenFailedSubjects",
        "WithdrawnSubjects"
    )
)

# Recalculate the same counts directly from Silver subjects
display(
    subjects
    .filter(F.col("SiteID") == "SITE030")
    .groupBy("SiteID")
    .agg(
        F.count("*").alias("RecalculatedTotalSubjects"),
        F.sum(F.when(F.col("SubjectStatus") == "Enrolled", 1).otherwise(0))
            .alias("RecalculatedEnrolledSubjects"),
        F.sum(F.when(F.col("SubjectStatus") == "Active", 1).otherwise(0))
            .alias("RecalculatedActiveSubjects"),
        F.sum(F.when(F.col("SubjectStatus") == "Completed", 1).otherwise(0))
            .alias("RecalculatedCompletedSubjects"),
        F.sum(F.when(F.col("SubjectStatus") == "Screen Failed", 1).otherwise(0))
            .alias("RecalculatedScreenFailedSubjects"),
        F.sum(F.when(F.col("SubjectStatus") == "Withdrawn", 1).otherwise(0))
            .alias("RecalculatedWithdrawnSubjects")
    )
)

SiteID,TotalSubjects,EnrolledSubjects,ActiveSubjects,CompletedSubjects,ScreenFailedSubjects,WithdrawnSubjects
SITE030,22,20,11,4,1,3


SiteID,RecalculatedTotalSubjects,RecalculatedEnrolledSubjects,RecalculatedActiveSubjects,RecalculatedCompletedSubjects,RecalculatedScreenFailedSubjects,RecalculatedWithdrawnSubjects
SITE030,22,2,11,4,1,3


In [0]:
%sql
SELECT *
FROM gpo_analytics.silver.visit
LIMIT 10;

VisitID,SubjectID,VisitName,VisitDate,Status
V0000001,SUB00001,Screening,2026-01-22,Completed
V0000002,SUB00001,Baseline,2026-01-29,Cancelled
V0000003,SUB00001,Week 4,2026-02-19,Completed
V0000004,SUB00001,Week 8,2026-03-19,Completed
V0000005,SUB00001,Week 12,2026-04-16,Completed
V0000006,SUB00002,Screening,2026-09-18,Completed
V0000007,SUB00002,Baseline,2026-09-25,Completed
V0000008,SUB00003,Screening,2025-06-15,Completed
V0000009,SUB00003,Baseline,2025-06-22,Scheduled
V0000010,SUB00003,Week 4,2025-07-13,Missed


In [0]:
%sql
CREATE OR REPLACE TABLE gpo_analytics.gold.patient_visits AS
SELECT
    COUNT(DISTINCT VisitID) AS TotalVisits,
    COUNT(DISTINCT SubjectID) AS SubjectsWithVisits,
    COUNT_IF(Status = 'Completed') AS CompletedVisits,
    COUNT_IF(Status = 'Missed') AS MissedVisits,
    COUNT_IF(Status = 'Scheduled') AS ScheduledVisits,
    COUNT_IF(Status = 'Cancelled') AS CancelledVisits
FROM gpo_analytics.silver.visit;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT *
FROM gpo_analytics.gold.patient_visits;

TotalVisits,SubjectsWithVisits,CompletedVisits,MissedVisits,ScheduledVisits,CancelledVisits
14398,3000,9799,1313,2591,695


In [0]:
%sql
SHOW TABLES IN gpo_analytics.silver;

database,tableName,isTemporary
silver,adverseevents,false
silver,drugdispensation,false
silver,investigator,false
silver,labresults,false
silver,randomization,false
silver,shipment,false
silver,site,false
silver,study,false
silver,subject,false
silver,visit,false


In [0]:
%sql
DESCRIBE TABLE gpo_analytics.silver.visit;

col_name,data_type,comment
VisitID,string,null
SubjectID,string,null
VisitName,string,null
VisitDate,date,null
Status,string,null


In [0]:
%sql
DESCRIBE TABLE gpo_analytics.silver.subject;

col_name,data_type,comment
SubjectID,string,null
StudyID,string,null
SiteID,string,null
Gender,string,null
Age,int,null
Disease,string,null
ScreenDate,date,null
SubjectStatus,string,null
EnrollmentDate,date,null


In [0]:
%sql
SELECT
    s.StudyID,
    COUNT(v.VisitID) AS TotalVisits,
    SUM(CASE WHEN LOWER(TRIM(v.Status)) = 'completed' THEN 1 ELSE 0 END) AS CompletedVisits,
    SUM(CASE WHEN LOWER(TRIM(v.Status)) = 'missed' THEN 1 ELSE 0 END) AS MissedVisits,
    SUM(CASE WHEN LOWER(TRIM(v.Status)) = 'scheduled' THEN 1 ELSE 0 END) AS ScheduledVisits,
    SUM(CASE WHEN LOWER(TRIM(v.Status)) = 'cancelled' THEN 1 ELSE 0 END) AS CancelledVisits
FROM gpo_analytics.silver.subject s
LEFT JOIN gpo_analytics.silver.visit v
    ON s.SubjectID = v.SubjectID
GROUP BY s.StudyID
ORDER BY s.StudyID;

StudyID,TotalVisits,CompletedVisits,MissedVisits,ScheduledVisits,CancelledVisits
ST001,483,341,34,87,21
ST002,888,598,78,168,44
ST003,591,393,60,106,32
ST004,550,379,43,105,23
ST005,506,347,53,83,23
ST006,583,400,42,112,29
ST007,873,592,84,151,46
ST008,225,163,17,40,5
ST009,993,654,101,184,54
ST010,1013,679,93,191,50


In [0]:
%sql
CREATE OR REPLACE TABLE gpo_analytics.gold.study_visit_summary AS
SELECT
    s.StudyID,
    COUNT(v.VisitID) AS TotalVisits,
    SUM(CASE WHEN LOWER(TRIM(v.Status)) = 'completed' THEN 1 ELSE 0 END) AS CompletedVisits,
    SUM(CASE WHEN LOWER(TRIM(v.Status)) = 'missed' THEN 1 ELSE 0 END) AS MissedVisits,
    SUM(CASE WHEN LOWER(TRIM(v.Status)) = 'scheduled' THEN 1 ELSE 0 END) AS ScheduledVisits,
    SUM(CASE WHEN LOWER(TRIM(v.Status)) = 'cancelled' THEN 1 ELSE 0 END) AS CancelledVisits
FROM gpo_analytics.silver.subject s
LEFT JOIN gpo_analytics.silver.visit v
    ON s.SubjectID = v.SubjectID
GROUP BY s.StudyID;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT *
FROM gpo_analytics.gold.study_visit_summary
ORDER BY StudyID;

StudyID,TotalVisits,CompletedVisits,MissedVisits,ScheduledVisits,CancelledVisits
ST001,483,341,34,87,21
ST002,888,598,78,168,44
ST003,591,393,60,106,32
ST004,550,379,43,105,23
ST005,506,347,53,83,23
ST006,583,400,42,112,29
ST007,873,592,84,151,46
ST008,225,163,17,40,5
ST009,993,654,101,184,54
ST010,1013,679,93,191,50


In [0]:
%sql
DESCRIBE gpo_analytics.silver.shipment;

col_name,data_type,comment
ShipmentID,string,null
LotNumber,string,null
Depot,string,null
SiteID,string,null
ShipmentDate,date,null
Status,string,null
ExpectedDeliveryDate,date,null
ActualDeliveryDate,date,null


In [0]:
%sql
SELECT DISTINCT Status
FROM gpo_analytics.silver.shipment
ORDER BY Status;

Status
Delayed
Delivered
In Transit
Pending
Returned


In [0]:
%sql
CREATE OR REPLACE TABLE gpo_analytics.gold.shipment_summary AS
SELECT
    COUNT(ShipmentID) AS TotalShipments,
    SUM(CASE WHEN LOWER(TRIM(Status)) = 'delivered' THEN 1 ELSE 0 END) AS DeliveredShipments,
    SUM(CASE WHEN LOWER(TRIM(Status)) = 'pending' THEN 1 ELSE 0 END) AS PendingShipments,
    SUM(CASE WHEN LOWER(TRIM(Status)) = 'in transit' THEN 1 ELSE 0 END) AS InTransitShipments,
    SUM(CASE WHEN LOWER(TRIM(Status)) = 'delayed' THEN 1 ELSE 0 END) AS DelayedShipments,
    SUM(CASE WHEN LOWER(TRIM(Status)) = 'returned' THEN 1 ELSE 0 END) AS ReturnedShipments
FROM gpo_analytics.silver.shipment;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT *
FROM gpo_analytics.gold.shipment_summary;

TotalShipments,DeliveredShipments,PendingShipments,InTransitShipments,DelayedShipments,ReturnedShipments
600,381,69,91,50,9
